<a href="https://colab.research.google.com/github/peperibeirolopes-bot/Mission-Control-AI-Space-tech-/blob/main/GS_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 0. Instalar zstd (necessário para Ollama)
!apt-get update && apt-get install -y zstd

# 1. Instalar o Ollama no Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Iniciar o servidor em background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 3. Baixar o modelo Llama 3.1 1B
!ollama pull llama3.2:1b

# 4. Usar o modelo via biblioteca ollama
!pip install ollama -q

import ollama

resposta = ollama.chat(
model="llama3.2:1b",
messages=[
    {
        "role": "system",
        "content": "Você é um sistema de controle de missão espacial. Analise os dados e indique alertas."
    },
    {
      "role": "user",
      "content": "Temperatura: 95°C | Energia: 18% | Comunicação: instável. Avalie o status."
    }
  ]
)

print(resposta["message"]["content"])

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,695 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

In [4]:
import random
import ollama

# =========================
# CONFIGURAÇÃO DA IA
# =========================
SYSTEM_PROMPT = """
Você é o Mission Control AI, um chatbot de missão espacial.

Sua função é responder perguntas usando SOMENTE os dados recebidos.

Regras:
- Não invente informações
- Não explique regras internas
- Não crie novos dados
- Não contradiga o status oficial
- Seja direto
- Responda em português do Brasil

Use exatamente os dados fornecidos no contexto.
"""

# =========================
# FUNÇÕES DO SISTEMA
# =========================
def gerar_dados():
    return {
        "temperatura": random.randint(60,100),
        "energia": random.randint(20,100),
        "comunicacao": random.choices(
            ["estável","instável","crítica"],
            weights=[70,25,5]
        )[0]
    }

def gerar_alertas(dados):
    alertas = []

    if dados["temperatura"] > 90:
        alertas.append("Temperatura elevada")
    if dados["energia"] < 20:
        alertas.append("Energia crítica")
    if dados["comunicacao"] == "crítica":
        alertas.append("Falha de comunicação")

    return alertas

def decidir(dados):

    if dados["energia"] < 20:
        return "Ativar modo economia"

    if dados["temperatura"] > 100:
        return "Desligar módulos não essenciais"

    if dados["temperatura"] > 90:
        return "Reduzir atividade dos sistemas"

    if dados["comunicacao"] == "crítica":
        return "Reiniciar sistema de comunicação"

    return "Operação normal"

def status_missao(dados):
    if (
        dados["temperatura"] > 100
        or dados["energia"] < 20
        or dados["comunicacao"] == "crítica"
    ):
        return "CRÍTICO"

    elif (
        dados["temperatura"] > 90
        or dados["energia"] < 40
        or dados["comunicacao"] == "instável"
    ):
        return "ALERTA"

    return "OK"

def mostrar_estado(dados, status, alertas, acao):
    print("\n" + "=" * 50)
    print("🚀 MISSION CONTROL AI")
    print("=" * 50)
    print(f"🌡 Temperatura : {dados['temperatura']}°C")
    print(f"🔋 Energia     : {dados['energia']}%")
    print(f"📡 Comunicação : {dados['comunicacao']}")
    print(f"🧭 Status      : {status}")
    print(f"⚠ Alertas     : {alertas if alertas else 'Nenhum'}")
    print(f"⚙ Ação        : {acao}")
    print("=" * 50)

def analisar_ia(dados, alertas, acao, status, pergunta):
    try:
        resposta = ollama.chat(
            model="llama3.2:1b",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"""
Temperatura: {dados["temperatura"]}°C
Energia: {dados["energia"]}%
Comunicação: {dados["comunicacao"]}

Status oficial: {status}
Alertas oficiais: {alertas}
Ação automática: {acao}

Pergunta do operador:
{pergunta}
"""
                }
            ]
        )
        return resposta["message"]["content"].strip()
    except Exception as e:
        return f"Erro ao consultar a IA: {e}"

# =========================
# RESPOSTAS DIRETAS DO SISTEMA
# =========================
def responder_direto(pergunta, dados, status, alertas, acao):
    texto = pergunta.lower().strip()

    if "status" in texto:
        print("\n🤖 Mission Control:")
        print(f"Status: {status}")
        print(f"Alertas: {alertas if alertas else 'Nenhum'}")
        print(f"Recomendação: {acao}")
        return True

    if "alerta" in texto:
        print("\n🤖 Mission Control:")
        if alertas:
            print(f"Alertas ativos: {alertas}")
        else:
            print("Nenhum alerta ativo")
        return True

    if "recomenda" in texto or "recomendação" in texto:
        print("\n🤖 Mission Control:")
        print(f"Recomendação: {acao}")
        return True

    if "segura" in texto:
        print("\n🤖 Mission Control:")
        if status == "OK":
            print("Missão segura.")
        elif status == "ALERTA":
            print("Missão requer atenção.")
        else:
            print("Missão em estado crítico.")
        return True

    if "temperatura" in texto:
        print("\n🤖 Mission Control:")
        print(f"Temperatura atual: {dados['temperatura']}°C")
        return True

    if "energia" in texto:
        print("\n🤖 Mission Control:")
        print(f"Energia atual: {dados['energia']}%")
        return True

    if "comunicação" in texto or "comunicacao" in texto:
        print("\n🤖 Mission Control:")
        print(f"Comunicação atual: {dados['comunicacao']}")
        return True

    return False

# =========================
# INICIALIZAÇÃO
# =========================
dados = gerar_dados()
alertas = gerar_alertas(dados)
acao = decidir(dados)
status = status_missao(dados)

print("🚀 Mission Control AI iniciado.")
print("Digite 'nova missão' para gerar novos dados.")
print("Digite 'sair' para encerrar.\n")

mostrar_estado(dados, status, alertas, acao)

# =========================
# LOOP DO CHATBOT
# =========================
while True:
    pergunta = input("\nVocê: ")

    if pergunta.lower().strip() == "sair":
        print("\n🚀 Encerrando Mission Control AI...")
        print("📡 Comunicação encerrada.")
        print("👋 Até a próxima missão.")
        break

    if pergunta.lower().strip() == "nova missão":
        dados = gerar_dados()
        alertas = gerar_alertas(dados)
        acao = decidir(dados)
        status = status_missao(dados)

        print("\n🚀 Nova missão gerada")
        print(dados)
        print("🧭 Status:", status)
        continue

    if responder_direto(pergunta, dados, status, alertas, acao):
        continue

    resposta_ia = analisar_ia(dados, alertas, acao, status, pergunta)

    print("\n🤖 Mission Control:")
    print(resposta_ia)

🚀 Mission Control AI iniciado.
Digite 'nova missão' para gerar novos dados.
Digite 'sair' para encerrar.


🚀 MISSION CONTROL AI
🌡 Temperatura : 89°C
🔋 Energia     : 42%
📡 Comunicação : estável
🧭 Status      : OK
⚠ Alertas     : Nenhum
⚙ Ação        : Operação normal

Você: qual o status da missão?

🤖 Mission Control:
Status: OK
Alertas: Nenhum
Recomendação: Operação normal

Você: existem alertas ativos?

🤖 Mission Control:
Nenhum alerta ativo

Você: como está a comunicação?

🤖 Mission Control:
Comunicação atual: estável

Você: a missão está segura?

🤖 Mission Control:
Missão segura.

Você: qual o maior problema agora?

🤖 Mission Control:
Nenhuma.

Você: sair

🚀 Encerrando Mission Control AI...
📡 Comunicação encerrada.
👋 Até a próxima missão.
